# Lab 10: Multi-Layer Perceptron (MLP) for Time Series Forecasting

**University of Engineering and Technology Peshawar, Nowshera Campus**

**Course:** Machine Learning Lab

**Student Name:** Muhammad Ayub  
**Registration Number:** 22jzele0470

**Date:** May 14, 2026

---

## Objective
Build and train a Multi-Layer Perceptron (MLP) model for univariate multi-step time series forecasting using the AEP hourly energy dataset.

## Table of Contents
1. [Imports](#imports)
2. [Model Architecture](#model)
3. [Callbacks & Training Setup](#callbacks)
4. [Data Loading & Preparation](#data)
5. [Model Training](#training)
6. [Evaluation](#evaluation)
7. [Fine Tuning](#finetune)
8. [Conclusion](#conclusion)

## 1. Imports <a id='imports'></a>

In [2]:
import os
import numpy as np
import pandas as pd
import pickle
import time
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.callbacks.TrainingMonitor import TrainingMonitor

## 2. Model Architecture <a id='model'></a>

In [3]:
time_steps = 24
num_features = 21

def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps, num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

model = MLP()
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 504)               0         
                                                                 
 dense (Dense)               (None, 32)                16160     
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 16193 (63.25 KB)
Trainable params: 16193 (63.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


## 3. Callbacks & Training Setup <a id='callbacks'></a>

In [4]:
OUTPUT_PATH = r'C:\Users\M Ayub\Downloads\ML_LAB\lab_10'
checkpoints = os.path.join(OUTPUT_PATH, 'E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5')
FIG_PATH = os.path.join(OUTPUT_PATH, 'history.png')
JSON_PATH = os.path.join(OUTPUT_PATH, 'history.json')

EpochCheckpoint1 = ModelCheckpoint(checkpoints, monitor="val_loss", save_best_only=True, verbose=1)
TrainingMonitor1 = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=0)

callbacks = [EpochCheckpoint1, TrainingMonitor1]

## 4. Data Loading & Preparation <a id='data'></a>

In [5]:
path_dataset = r'C:\Users\M Ayub\Downloads\ML_LAB'

df_tr = pd.read_csv(os.path.join(path_dataset, 'train.csv'))
df_v  = pd.read_csv(os.path.join(path_dataset, 'validation.csv'))
df_te = pd.read_csv(os.path.join(path_dataset, 'test.csv'))

train_set = df_tr.values
validation_set = df_v.values
test_set = df_te.values

scaler = pickle.load(open(os.path.join(path_dataset, 'AEP_scaler.pkl'), 'rb'))

print(train_set.shape, validation_set.shape, test_set.shape)

(860, 21) (90, 21) (30, 21)


c:\Users\M Ayub\anaconda3\envs\DSP\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Create sequences
train_X, train_y = univariate_multi_step(train_set, time_steps, target_col=0, target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0, target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0, target_len=1)

## 5. Model Training <a id='training'></a>

In [7]:
model = MLP()
opt = Adam(1e-3)
model.compile(loss='mae', optimizer=opt, metrics=["mae", "mape"])

History = model.fit(
    train_X, train_y,
    batch_size=32,
    epochs=60,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/60


18/27 [===================>..........] - ETA: 0s - loss: 0.1725 - mae: 0.1725 - mape: 99.9228  
Epoch 1: val_loss improved from inf to 0.08470, saving model to C:\Users\M Ayub\Downloads\ML_LAB\lab_10\E1-cp-0001-loss0.08.h5
27/27 [==============================] - 2s 26ms/step - loss: 0.1493 - mae: 0.1493 - mape: 83.9993 - val_loss: 0.0847 - val_mae: 0.0847 - val_mape: 29.5043
Epoch 2/60
23/27 [========================>.....] - ETA: 0s - loss: 0.0847 - mae: 0.0847 - mape: 46.5186

c:\Users\M Ayub\anaconda3\envs\DSP\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(



Epoch 2: val_loss improved from 0.08470 to 0.07050, saving model to C:\Users\M Ayub\Downloads\ML_LAB\lab_10\E1-cp-0002-loss0.07.h5
27/27 [==============================] - 1s 33ms/step - loss: 0.0820 - mae: 0.0820 - mape: 44.7764 - val_loss: 0.0705 - val_mae: 0.0705 - val_mape: 26.5538
Epoch 3/60
23/27 [========================>.....] - ETA: 0s - loss: 0.0717 - mae: 0.0717 - mape: 39.3771
Epoch 3: val_loss did not improve from 0.07050
27/27 [==============================] - 1s 32ms/step - loss: 0.0730 - mae: 0.0730 - mape: 38.8820 - val_loss: 0.0715 - val_mae: 0.0715 - val_mape: 27.3143
Epoch 4/60
25/27 [==========================>...] - ETA: 0s - loss: 0.0679 - mae: 0.0679 - mape: 33.3667
Epoch 4: val_loss improved from 0.07050 to 0.05353, saving model to C:\Users\M Ayub\Downloads\ML_LAB\lab_10\E1-cp-0004-loss0.05.h5


c:\Users\M Ayub\anaconda3\envs\DSP\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 0s 15ms/step - loss: 0.0677 - mae: 0.0677 - mape: 32.9024 - val_loss: 0.0535 - val_mae: 0.0535 - val_mape: 17.6193
Epoch 5/60
24/27 [=========================>....] - ETA: 0s - loss: 0.0543 - mae: 0.0543 - mape: 28.2892
Epoch 5: val_loss improved from 0.05353 to 0.05223, saving model to C:\Users\M Ayub\Downloads\ML_LAB\lab_10\E1-cp-0005-loss0.05.h5
27/27 [==============================] - 1s 21ms/step - loss: 0.0540 - mae: 0.0540 - mape: 27.8030 - val_loss: 0.0522 - val_mae: 0.0522 - val_mape: 19.5597


...

27/27 [==============================] - 0s 15ms/step - loss: 0.0304 - mae: 0.0304 - mape: 15.5226 - val_loss: 0.0252 - val_mae: 0.0252 - val_mape: 8.3774
Epoch 38/60
27/27 [==============================] - ETA: 0s - loss: 0.0341 - mae: 0.0341 - mape: 17.4027
Epoch 38: val_loss did not improve from 0.02519
27/27 [==============================] - 1s 23ms/step - loss: 0.0341 - mae: 0.0341 - mape: 17.4027 - val_loss: 0.0376 - val_mae: 0.0376 - val_mape: 10.6094
Epoch 39/60
20/27 [=====================>........] - ETA: 0s - loss: 0.0315 - mae: 0.0315 - mape: 17.8950
Epoch 39: val_loss did not improve from 0.02519
27/27 [==============================] - 0s 15ms/step - loss: 0.0306 - mae: 0.0306 - mape: 16.3855 - val_loss: 0.0275 - val_mae: 0.0275 - val_mape: 9.0160
Epoch 40/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0270 - mae: 0.0270 - mape: 13.1647
Epoch 40: val_loss did not improve from 0.02519
27/27 [==============================] - 0s 12ms/step - loss: 0.0271 - m

c:\Users\M Ayub\anaconda3\envs\DSP\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 37ms/step - loss: 0.0301 - mae: 0.0301 - mape: 15.0439 - val_loss: 0.0243 - val_mae: 0.0243 - val_mape: 7.4644
Epoch 43/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0279 - mae: 0.0279 - mape: 12.5363
Epoch 43: val_loss did not improve from 0.02431
27/27 [==============================] - 0s 14ms/step - loss: 0.0267 - mae: 0.0267 - mape: 12.3104 - val_loss: 0.0262 - val_mae: 0.0262 - val_mape: 7.8618
Epoch 44/60
19/27 [====================>.........] - ETA: 0s - loss: 0.0258 - mae: 0.0258 - mape: 11.3548
Epoch 44: val_loss did not improve from 0.02431
27/27 [==============================] - 0s 13ms/step - loss: 0.0254 - mae: 0.0254 - mape: 11.0129 - val_loss: 0.0342 - val_mae: 0.0342 - val_mape: 10.1285
Epoch 45/60
18/27 [===================>..........] - ETA: 0s - loss: 0.0244 - mae: 0.0244 - mape: 10.1977
Epoch 45: val_loss did not improve from 0.02431
27/27 [==============================] - 0s 12ms/step - loss: 0.0258 - m

c:\Users\M Ayub\anaconda3\envs\DSP\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 22ms/step - loss: 0.0306 - mae: 0.0306 - mape: 15.3023 - val_loss: 0.0241 - val_mae: 0.0241 - val_mape: 8.2672
Epoch 58/60
19/27 [====================>.........] - ETA: 0s - loss: 0.0273 - mae: 0.0273 - mape: 11.6813
Epoch 58: val_loss did not improve from 0.02406
27/27 [==============================] - 0s 17ms/step - loss: 0.0268 - mae: 0.0268 - mape: 11.6388 - val_loss: 0.0585 - val_mae: 0.0585 - val_mape: 17.1751
Epoch 59/60
18/27 [===================>..........] - ETA: 0s - loss: 0.0416 - mae: 0.0416 - mape: 22.6904
Epoch 59: val_loss did not improve from 0.02406
27/27 [==============================] - 0s 14ms/step - loss: 0.0389 - mae: 0.0389 - mape: 19.9446 - val_loss: 0.0422 - val_mae: 0.0422 - val_mape: 13.2226
Epoch 60/60
17/27 [=================>............] - ETA: 0s - loss: 0.0320 - mae: 0.0320 - mape: 13.9916
Epoch 60: val_loss did not improve from 0.02406
27/27 [==============================] - 1s 28ms/step - loss: 0.0290 - 

## 6. Evaluation <a id='evaluation'></a>

In [8]:
# Load best model
best_model_path = r'C:\Users\M Ayub\Downloads\ML_LAB\lab_10\E1-cp-0057-loss0.02.h5'
model = load_model(best_model_path)

y_pred_scaled = model.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

# Calculate metrics
MAE = np.mean(np.abs(y_pred - y_test_unscaled))
MEDAE = np.median(np.abs(y_pred - y_test_unscaled))
MSE = np.mean(np.square(y_pred - y_test_unscaled))
RMSE = np.sqrt(MSE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100

print(f"Mean Absolute Error (MAE): {MAE:.2f}")
print(f"Median Absolute Error (MedAE): {MEDAE:.2f}")
print(f"Root Mean Squared Error (RMSE): {RMSE:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {MAPE:.2f} %")
print(f"Median Absolute Percentage Error (MDAPE): {MDAPE:.2f} %")

1/1 [==============================] - 0s 237ms/step
Mean Absolute Error (MAE): 3254.93
Median Absolute Error (MedAE): 3079.47
Root Mean Squared Error (RMSE): 3745.44
Mean Absolute Percentage Error (MAPE): 20.95 %
Median Absolute Percentage Error (MDAPE): 19.90 %


## 7. Fine Tuning <a id='finetune'></a>

In [ ]:
# Fine tuning code (similar structure as above with lower learning rate)

## Conclusion <a id='conclusion'></a>

The MLP model was successfully trained for time series forecasting. Best model achieved good performance on the test set.

**GitHub Repository:**  
https://github.com/prince4775/8th-Semester-ML-and-DL-Lab